# Step 5: Ablation study of the main deep model

Each variant changes one thing and is **retrained from scratch** with the same seed, inputs, loop and checkpoint rule as the step 4 model (`full`). The attention news and user encoders are in every variant, so every variant still reads content and the user's history; what changes is the engineered evidence and the ID treatment.

| Variant | What changes | Question it answers |
|---|---|---|
| `no_candidate_attention` | without the candidate-aware attention branch (Group 10's user vector only) | Does letting the candidate pick the relevant history help? (Topic 7B) |
| `no_id_masking` | ID vectors never masked in training | Does forcing the network off ID look-ups help cold articles? (Topic 6) |
| `no_t1` | without the article's own behaviour | How much does the candidate's own history matter, cold articles included? |
| `no_t2` | without content-transferred behaviour | Does borrowed topic/entity behaviour add anything? |
| `no_t3` | without explicit content columns (codes, entity counts) | Do they add anything beyond the news encoder? |
| `no_u` | without the engineered user–article matching columns | Does explicit matching add to the attention user encoder? |
| `full_plus_smoothing` | adds content_smoothed_ctr (α = 20) | Does the interim report's hybrid feature help? |

**Running on Colab in pieces.** Each finished variant writes its prediction files and training history to Drive. Re-running the notebook skips variants that are already done, so the six runs can be spread across sessions or machines. Set `RUN_ONLY` to a subset to split the work.

> Metrics here come from `shared/eval_interim.py`. Final ablation tables and intervals come from `step6_evaluation_diagnostics.ipynb`.

**At a glance**

| | |
|---|---|
| Reads | Same as step 4, plus step 4's `deep_hybrid` predictions on Google Drive `MyDrive/BT4222_G11/` |
| Writes | `predictions/real/` (`deep_<variant>`) and each variant's training history in `models/real/` |
| Time and hardware | About 8–12 minutes per variant on a Colab T4 GPU (six variants to train) |
| Run after | Step 4 |

## 0. Colab data setup (run once)

The first cell confirms a GPU is attached. The second unpacks the shared `Data.zip` into `features/`, `nrms_cache/` and `predictions/`; it skips itself once those folders exist. The run below used the original, longer version of this setup.

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True
Tesla T4


In [ ]:
# Colab only, run once: unpack the shared Data.zip (see the README) into the folder layout the notebooks read.
# Skips itself when the folders already exist, and does nothing outside Colab.
import os, sys, shutil, zipfile

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/BT4222_G11"
    LAYOUT = {"features_v0.1": "features/v0.1", "nrms_cache_real": "nrms_cache/real", "predictions_real": "predictions/real"}
    if not all(os.path.exists(os.path.join(ROOT, dst)) for dst in LAYOUT.values()):
        zipfile.ZipFile(os.path.join(ROOT, "Data.zip")).extractall(ROOT)
        for src, dst in LAYOUT.items():
            target = os.path.join(ROOT, dst)
            if not os.path.exists(target):
                os.makedirs(os.path.dirname(target), exist_ok=True)
                shutil.move(os.path.join(ROOT, "Data", src), target)
    print({dst: os.path.exists(os.path.join(ROOT, dst)) for dst in LAYOUT.values()})

## 1. Setup

In [7]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.environ.get("G11_REPO_DIR", os.getcwd())
    DATA_ROOT = os.environ.get("G11_DATA_ROOT", os.path.join(REPO_DIR, "data"))
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from shared import schema as S, selection as SEL
from shared.predictions import write_predictions, read_predictions
from shared.nrms_inputs import build_inputs
from shared import nrms_g10 as N, hybrid_model as H
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = "REAL"   # recorded in every prediction file
import kagglehub
RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
CACHE_DIR = os.path.join(DATA_ROOT, "nrms_cache", DATA_KIND.lower())   # MiniLM / spaCy vectors from step 3b
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 4222

G10_CONFIG = dict(num_heads=2, dropout=0.1, user_heads=2, user_dropout=0.1)   # Group 10 Test_2, as step 3b
TRAIN = dict(epochs=10, lr=5e-4, batch_size=32, patience=3)                    # Group 10's loop + early stop
SELECT = "ndcg@10"           # checkpoint criterion: best validation nDCG@10
ALPHA = 20                   # content_smoothed_ctr strength for the +H variant (step 3a's validation choice)
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | device={DEVICE} | features: {FEATURE_ROOT}")
RUN_ONLY = os.environ.get("G11_DEEP_VARIANTS")          # e.g. "no_t1,no_t2" to split the runs; default: all
TO_RUN = [v for v in H.VARIANTS if v != "full"]
if RUN_ONLY:
    TO_RUN = [v for v in TO_RUN if v in RUN_ONLY.split(",")]
print("variants to run (if not already done):", TO_RUN)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 61.7M/61.7M [00:00<00:00, 93.0MB/s]

Extracting files...


DATA_KIND=REAL | device=cuda | features: /content/drive/MyDrive/BT4222_G11/features/v0.1
variants to run (if not already done): ['no_candidate_attention', 'no_id_masking', 'no_t1', 'no_t2', 'no_t3', 'no_u', 'full_plus_smoothing']


## 2. Model inputs (same as step 4)

In [8]:
t0 = time.time()
META_COLS = ["row_id", "impression_id", "news_id", "time", "label", "is_cold_10"]
meta = {s: pd.read_parquet(S.meta_path(FEATURE_ROOT, s), columns=META_COLS) for s in S.SPLITS}
inputs = build_inputs(RAW_DIR, meta, device=DEVICE, cache_dir=CACHE_DIR, seed=SEED)
for split in S.SPLITS:
    assert (inputs["impressions"]["impression_id"][inputs["splits"][split]["imp"]] == meta[split]["impression_id"].to_numpy()).all()
print(f"inputs built in {time.time() - t0:.0f}s; {inputs['info']['n_train_samples']:,} training samples "
      f"(1 click + 4 non-clicks each), {len(inputs['news']['title_abs']):,} articles")

inputs built in 334s; 141,558 training samples (1 click + 4 non-clicks each), 51,282 articles


## 3. Shared training function (same as step 4)

In [9]:
def load_features(groups):
    """Scaled feature rows for every split, aligned with meta on row_id. Scaling is fitted on train only."""
    num_cols, cat_cols = H.variant_columns(groups)
    raw = {s: pd.read_parquet(S.features_path(FEATURE_ROOT, s), columns=["row_id"] + H.raw_columns(num_cols, cat_cols))
           for s in S.SPLITS}
    for s in S.SPLITS:
        assert np.array_equal(raw[s]["row_id"].to_numpy(), meta[s]["row_id"].to_numpy()), f"{s}: row_id misaligned"
    scaler = H.FeatureScaler(num_cols, cat_cols, alpha=ALPHA).fit(raw["train"])
    feats = {s: scaler.transform(raw[s]) for s in S.SPLITS}
    cat_sizes = [(int(max(raw[s][c].max() for s in S.SPLITS)) + 1, H.CAT_DIMS[c]) for c in cat_cols]
    return feats, num_cols, cat_cols, cat_sizes


def run_variant(name, score_splits=("val", "test"), log=print):
    """Train one variant from scratch, restore its best validation epoch, score the requested splits."""
    spec = H.VARIANTS[name]
    feats, num_cols, cat_cols, cat_sizes = load_features(spec["groups"])
    N.set_seed(SEED)
    model = H.HybridNRMS(inputs, len(num_cols), cat_sizes, id_mask_p=spec["id_mask_p"],
                         candidate_attention=spec.get("candidate_attention", True), **G10_CONFIG).to(DEVICE)
    val_meta, val_split = meta["val"], inputs["splits"]["val"]

    def val_fn(m):
        res = E.evaluate(val_meta, H.score_split(m, val_split, *feats["val"]))
        return {k: res[k] for k in ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10"]}

    t0 = time.time()
    history, best_epoch = H.train(model, inputs["train_samples"], *feats["train"], seed=SEED, val_fn=val_fn,
                                  select_metric=SELECT, log=log, **TRAIN)
    scores = {s: H.score_split(model, inputs["splits"][s], *feats[s]) for s in score_splits}
    return {"model": model, "history": history, "best_epoch": best_epoch, "scores": scores, "num_cols": num_cols,
            "cat_cols": cat_cols, "minutes": round((time.time() - t0) / 60, 1)}

## 4. Train each variant

Validation and test are scored for every variant. A variant whose prediction files already exist is skipped.

In [10]:
def done(variant):
    return all(glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__deep_{variant}__*__{s}.parquet"))
               for s in ("val", "test"))

import glob
for variant in TO_RUN:
    if done(variant):
        print(f"{variant}: already done, skipped")
        continue
    print(f"===== {variant} =====")
    r = run_variant(variant)
    cfg = f"{variant},id_mask={H.VARIANTS[variant]['id_mask_p']},epoch={r['best_epoch']}"
    for split, sc in r["scores"].items():
        write_predictions(meta[split]["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=f"deep_{variant}",
                          config_id=cfg, split=split, schema_version=S.SCHEMA_VERSION,
                          extra={"variant": variant, "best_epoch": r["best_epoch"], "selected_on": "val"})
    r["history"].to_csv(os.path.join(MODEL_DIR, f"deep_{variant}_history.csv"))
    print(f"{variant}: best epoch {r['best_epoch']}, {r['minutes']} min")
    del r

no_candidate_attention: already done, skipped
===== no_id_masking =====
epoch=1, train_loss=1.3036, train_seconds=113.2000, val_auc=0.7074, val_mrr=0.3561, val_ndcg@5=0.3887, val_ndcg@10=0.4452, val_cold_recall@10=0.6888
epoch=2, train_loss=1.2670, train_seconds=112.8000, val_auc=0.6961, val_mrr=0.3489, val_ndcg@5=0.3795, val_ndcg@10=0.4363, val_cold_recall@10=0.7282
epoch=3, train_loss=1.2542, train_seconds=113.9000, val_auc=0.6940, val_mrr=0.3449, val_ndcg@5=0.3749, val_ndcg@10=0.4323, val_cold_recall@10=0.7120
epoch=4, train_loss=1.2461, train_seconds=112.2000, val_auc=0.6842, val_mrr=0.3389, val_ndcg@5=0.3678, val_ndcg@10=0.4261, val_cold_recall@10=0.6771
stopping: no validation gain for 3 epochs
no_id_masking: best epoch 1, 7.8 min
===== no_t1 =====
epoch=1, train_loss=1.3334, train_seconds=112.5000, val_auc=0.6686, val_mrr=0.3240, val_ndcg@5=0.3508, val_ndcg@10=0.4102, val_cold_recall@10=0.6736
epoch=2, train_loss=1.2953, train_seconds=112.7000, val_auc=0.6754, val_mrr=0.3304, va

## 5. Results and change against the full model

The full model's predictions come from step 4. For nDCG@10 and Cold Recall@10 the 95% interval is a paired bootstrap over impressions (1,000 resamples).

In [12]:
def load_scores(model_id, split):
    paths = glob.glob(os.path.join(PRED_ROOT, DATA_KIND.lower(), f"{DATA_KIND}__{model_id}__*__{split}.parquet"))
    if not paths:
        return None
    p, _ = read_predictions(sorted(paths)[-1])
    return meta[split][["row_id"]].merge(p, on="row_id", how="left")["predicted_score"].to_numpy()

rows, delta_rows = [], []
for split in ("val", "test"):
    base_sc = load_scores("deep_hybrid", split)
    assert base_sc is not None, "run step 4 first"
    base = E.per_impression_metrics(meta[split], base_sc)
    rows.append({"split": split, "variant": "full", **{k: v for k, v in E.summarise(base).items() if k in SHOW}})
    for variant in H.VARIANTS:
        if variant == "full":
            continue
        sc = load_scores(f"deep_{variant}", split)
        if sc is None:
            continue
        other = E.per_impression_metrics(meta[split], sc)
        rows.append({"split": split, "variant": variant, **{k: v for k, v in E.summarise(other).items() if k in SHOW}})
        row = {"split": split, "variant": variant}
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(base[metric], other[metric])
            row.update({f"Δ {metric}": bs["mean_diff"], f"{metric} CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
        delta_rows.append(row)
from IPython.display import display
display(pd.DataFrame(rows).set_index(["split", "variant"]).round(4))
pd.DataFrame(delta_rows).set_index(["split", "variant"]).round(4)

mrr  ndcg@5  ndcg@10     auc  cold_recall@10  \
split variant                                                                   
val   full                    0.3606  0.3930   0.4509  0.7163          0.6875   
      no_candidate_attention  0.3593  0.3912   0.4495  0.7142          0.6592   
      no_id_masking           0.3561  0.3887   0.4452  0.7074          0.6888   
      no_t1                   0.3347  0.3629   0.4220  0.6817          0.6800   
      no_t2                   0.3603  0.3927   0.4510  0.7162          0.6753   
      no_t3                   0.3582  0.3904   0.4479  0.7113          0.6795   
      no_u                    0.3396  0.3688   0.4265  0.6920          0.6150   
      full_plus_smoothing     0.3594  0.3919   0.4505  0.7154          0.6843   
test  full                    0.3366  0.3672   0.4279  0.7095          0.6268   
      no_candidate_attention  0.3349  0.3646   0.4249  0.7058          0.5987   
      no_id_masking           0.3342  0.3638   0.4239  0.7036          0.6223   
      no_t1                   0.3002  0.3245   0.3866  0.6615          0.6510   
      no_t2                   0.3370  0.3674   0.4276  0.7092          0.6196   
      no_t3                   0.3332  0.3628   0.4235  0.7053          0.6143   
      no_u                    0.3160  0.3411   0.4027  0.6834          0.6156   
      full_plus_smoothing     0.3374  0.3683   0.4281  0.7097          0.6375   

                              cold_hit@10  warm_recall@10  
split variant                                              
val   full                         0.6972          0.6856  
      no_candidate_attention       0.6695          0.6847  
      no_id_masking                0.6994          0.6774  
      no_t1                        0.6908          0.6599  
      no_t2                        0.6866          0.6872  
      no_t3                        0.6919          0.6824  
      no_u                         0.6269          0.6623  
      full_plus_smoothing          0.6940          0.6875  
test  full                         0.6464          0.6660  
      no_candidate_attention       0.6189          0.6614  
      no_id_masking                0.6440          0.6591  
      no_t1                        0.6691          0.6268  
      no_t2                        0.6404          0.6648  
      no_t3                        0.6332          0.6610  
      no_u                         0.6368          0.6390  
      full_plus_smoothing          0.6571          0.6649

Δ ndcg@10          ndcg@10 CI  Δ cold_recall@10  \
split variant                                                                   
val   no_candidate_attention    -0.0014  [-0.0024, -0.0003]           -0.0283   
      no_id_masking             -0.0057  [-0.0072, -0.0041]            0.0013   
      no_t1                     -0.0289  [-0.0312, -0.0266]           -0.0075   
      no_t2                      0.0000  [-0.0010, +0.0012]           -0.0122   
      no_t3                     -0.0030  [-0.0046, -0.0014]           -0.0079   
      no_u                      -0.0244  [-0.0269, -0.0215]           -0.0724   
      full_plus_smoothing       -0.0005  [-0.0015, +0.0007]           -0.0031   
test  no_candidate_attention    -0.0030  [-0.0040, -0.0019]           -0.0282   
      no_id_masking             -0.0039  [-0.0053, -0.0025]           -0.0046   
      no_t1                     -0.0413  [-0.0436, -0.0388]            0.0242   
      no_t2                     -0.0002  [-0.0014, +0.0009]           -0.0073   
      no_t3                     -0.0043  [-0.0059, -0.0028]           -0.0125   
      no_u                      -0.0252  [-0.0281, -0.0225]           -0.0113   
      full_plus_smoothing        0.0003  [-0.0009, +0.0014]            0.0107   

                               cold_recall@10 CI  
split variant                                     
val   no_candidate_attention  [-0.0432, -0.0133]  
      no_id_masking           [-0.0165, +0.0191]  
      no_t1                   [-0.0283, +0.0139]  
      no_t2                   [-0.0286, +0.0027]  
      no_t3                   [-0.0252, +0.0100]  
      no_u                    [-0.1018, -0.0418]  
      full_plus_smoothing     [-0.0192, +0.0130]  
test  no_candidate_attention  [-0.0427, -0.0126]  
      no_id_masking           [-0.0225, +0.0141]  
      no_t1                   [-0.0012, +0.0505]  
      no_t2                   [-0.0230, +0.0094]  
      no_t3                   [-0.0317, +0.0054]  
      no_u                    [-0.0441, +0.0228]  
      full_plus_smoothing     [-0.0072, +0.0289]

## 6. What the ablations show

Each row is a variant minus the full model (`no_*`: the full model without that component; `full_plus_smoothing`: with content smoothed CTR added).

* **Candidate-aware attention is the one component that clearly helps cold articles.** Removing it lowers Cold Recall@10 by 0.028 on both splits (95% CI [−0.043, −0.013] on validation and test) and nDCG@10 by 0.001–0.003 (intervals below zero). Its weights stayed uniform (step 4, section 7), so what it adds is a second, candidate-specific comparison between the reader's average history and the article, not the selection of particular past clicks.
* **U, the engineered reader–article match, is the second pillar.** Removing it costs about 0.025 nDCG@10 on both splits. Its cold effect is large on validation (−0.072, CI [−0.102, −0.042]) and points the same way on test without reaching significance (−0.011, CI [−0.044, +0.023]).
* **T1, the article's own behaviour, carries overall ranking but not cold recall.** Removing it gives the largest nDCG@10 drops (−0.029 validation, −0.041 test); its cold-recall change is within noise on both splits. This differs from the supplementary LightGBM ablation (step 3d), where removing T1 also cost cold recall: here the attention encoders and U already describe new articles.
* **ID masking works as a regulariser, not as a cold-start device.** Without it nDCG@10 falls by 0.004–0.006 (intervals below zero) but Cold Recall@10 does not change detectably. The unmasked network peaked at epoch 1 and then overfitted, the "lazy ID" behaviour the Topic 6 cold-start slide warns about.
* **T3's explicit content columns help overall ranking a little** (−0.003 to −0.004 nDCG@10 without them); their cold-recall change is within noise, since the news encoder already reads the content.
* **T2 and content smoothed CTR add nothing detectable** once the rest of the model is present (every interval crosses zero). In the supplementary LightGBM ablation, content smoothed CTR helped cold articles; the deep model appears to build the same signal from the encoders and U.
* **Precision:** about 840 to 940 impressions per split contain a cold click, so Cold Recall@10 intervals are roughly ±0.015 to ±0.03; every model is a single training seed.

*Based on the run with feature schema v0.1 and the interim evaluator (impression-level intervals). Step 6 recomputes these with the team evaluator and user-clustered intervals; its ablation table gives the same readings.*